In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
print(os.listdir('.'))

['.config', 'drive', 'sample_data']


In [4]:
import os
print(os.listdir('.'))
os.chdir('drive/My Drive/Projects/Kaggle/NVIDIA_Nemotron/NVIDIA_Nemotron')
print(os.listdir('.'))

['.config', 'drive', 'sample_data']
['.git', 'main.py', 'adapter', 'nvidia-nemotron-training.ipynb', 'basic.py', 'basic.ipynb']


In [5]:
import csv
import os
import sys
from pathlib import Path


DEFAULT_MODEL_ID = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16"
#SCRIPT_DIR = Path(__file__).resolve().parent
SCRIPT_DIR = Path.cwd()
DEFAULT_CSV_PATH = (SCRIPT_DIR / ".." / "data" / "train.csv").resolve()
DEFAULT_MODEL_DIR = (SCRIPT_DIR / ".." / "models" / DEFAULT_MODEL_ID.replace("/", "--")).resolve()
#DEFAULT_MODEL_DIR = (SCRIPT_DIR / "models" / DEFAULT_MODEL_ID.replace("/", "--")).resolve()


# Edit these values, then run each cell below in order.
MODEL_ID = DEFAULT_MODEL_ID
CSV_PATH = DEFAULT_CSV_PATH
PROMPT_COLUMN = "prompt"
NUM_EXAMPLES = 3
DOWNLOAD_DIR = DEFAULT_MODEL_DIR
# set HF_TOKEN env var or HUGGINGFACE_HUB_TOKEN env var to avoid download prompts

os.environ["HF_TOKEN"] = 
HF_TOKEN = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")

MAX_INPUT_TOKENS = 2048
MAX_NEW_TOKENS = 128
DEVICE_MAP = "auto"
DTYPE = "auto"  # auto, bfloat16, float16, float32
DO_SAMPLE = False
TEMPERATURE = 0.7


def import_dependencies():
	try:
		import torch
		from huggingface_hub import snapshot_download
		from transformers import AutoModelForCausalLM, AutoTokenizer
	except ImportError as exc:
		raise SystemExit(
			"Missing dependencies. Install them with: pip install torch transformers huggingface_hub accelerate"
		) from exc

	return torch, snapshot_download, AutoModelForCausalLM, AutoTokenizer


def load_prompts(csv_path: Path, prompt_column: str, num_examples: int) -> list[str]:
	if num_examples <= 0:
		raise ValueError("NUM_EXAMPLES must be greater than 0.")

	if not csv_path.exists():
		raise FileNotFoundError(f"CSV file not found: {csv_path}")

	prompts: list[str] = []
	with csv_path.open("r", encoding="utf-8", newline="") as handle:
		reader = csv.DictReader(handle)
		if reader.fieldnames is None or prompt_column not in reader.fieldnames:
			raise KeyError(
				f"Column '{prompt_column}' not found in {csv_path}. Available columns: {reader.fieldnames}"
			)

		for row in reader:
			value = (row.get(prompt_column) or "").strip()
			if not value:
				continue
			prompts.append(value)
			if len(prompts) >= num_examples:
				break

	if not prompts:
		raise ValueError(f"No non-empty prompts found in column '{prompt_column}' of {csv_path}.")

	return prompts


def download_model(snapshot_download, model_id: str, download_dir: Path, hf_token: str | None) -> Path:
	download_dir.mkdir(parents=True, exist_ok=True)
	local_path = snapshot_download(
		repo_id=model_id,
		local_dir=str(download_dir),
		token=hf_token,
		resume_download=True,
	)
	return Path(local_path).resolve()


def resolve_dtype(torch_module, dtype_name: str):
	if dtype_name == "auto":
		return torch_module.bfloat16 if torch_module.cuda.is_available() else torch_module.float32
	return getattr(torch_module, dtype_name)


def resolve_input_device(torch_module, model) -> str:
	if torch_module.cuda.is_available():
		return "cuda"

	try:
		return str(next(model.parameters()).device)
	except StopIteration:
		return "cpu"


def build_generate_kwargs(max_new_tokens: int, do_sample: bool, temperature: float, tokenizer) -> dict:
	kwargs = {
		"max_new_tokens": max_new_tokens,
		"do_sample": do_sample,
		"pad_token_id": tokenizer.eos_token_id,
		"eos_token_id": tokenizer.eos_token_id,
	}
	if do_sample:
		kwargs["temperature"] = temperature
	return kwargs


def generate_completion(torch_module, model, tokenizer, prompt: str) -> str:
	tokenized = tokenizer(
		prompt,
		return_tensors="pt",
		truncation=True,
		max_length=MAX_INPUT_TOKENS,
	)
	tokenized = tokenized.to(resolve_input_device(torch_module, model))

	with torch_module.no_grad():
		outputs = model.generate(
			**tokenized,
			**build_generate_kwargs(MAX_NEW_TOKENS, DO_SAMPLE, TEMPERATURE, tokenizer),
		)

	prompt_length = tokenized["input_ids"].shape[1]
	return tokenizer.decode(outputs[0][prompt_length:], skip_special_tokens=True).strip()


def show_examples(torch_module, model, tokenizer, prompts: list[str]) -> None:
	for index, prompt in enumerate(prompts, start=1):
		completion = generate_completion(torch_module, model, tokenizer, prompt)
		print(f"\n=== Example {index} ===")
		print("Prompt:")
		print(prompt)
		print("\nCompletion:")
		print(completion or "<empty completion>")



In [6]:
torch, snapshot_download, AutoModelForCausalLM, AutoTokenizer = import_dependencies()
torch_dtype = resolve_dtype(torch, DTYPE)

In [7]:
if not torch.cuda.is_available():
	print(
		"Warning: CUDA was not detected. Loading this 30B model on CPU is likely impractical.",
		file=sys.stderr,
		flush=True,
	)

In [8]:

prompts = load_prompts(CSV_PATH.resolve(), PROMPT_COLUMN, NUM_EXAMPLES)
print(f"Loaded {len(prompts)} prompts from {CSV_PATH}")
prompts


Loaded 3 prompts from /content/drive/My Drive/Projects/Kaggle/NVIDIA_Nemotron/data/train.csv


["In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n01010001 -> 11011101\n00001001 -> 01101101\n00010101 -> 01010101\n11111111 -> 10000001\n10011101 -> 01000101\n00111011 -> 00001001\n10111101 -> 00000101\n00100110 -> 10110011\n\nNow, determine the output for: 00110100",
 "In Alice's Wonderland, a secret bit manipulation rule transforms 8-bit binary numbers. The transformation involves operations like bit shifts, rotations, XOR, AND, OR, NOT, and possibly majority or choice functions.\n\nHere are some examples of input -> output:\n10001110 -> 00100110\n10011001 -> 01000100\n01100100 -> 00010001\n10000010 -> 00001010\n00011011 -> 01001100\n00111010 -> 10011100\n01101111 -> 00110111\n10010110 -> 01011010\n00001010 -> 00101100\n\nNow, determine the output for: 11100000",
 "In Ali

In [13]:
local_model_dir = '../models/nvidia--NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16'

In [14]:
print("Loading tokenizer...", flush=True)
tokenizer = AutoTokenizer.from_pretrained(local_model_dir, trust_remote_code=True)


Loading tokenizer...


In [2]:
!pip uninstall -y mamba-ssm causal-conv1d

In [2]:
!nvidia-smi


Mon Mar 23 17:41:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip list | grep torch

torch                                    2.10.0+cu128
torchao                                  0.10.0
torchaudio                               2.10.0+cu128
torchcodec                               0.10.0+cu128
torchdata                                0.11.0
torchsummary                             1.5.1
torchtune                                0.6.1
torchvision                              0.25.0+cu128


In [6]:
!pip uninstall torch torchvision torchaudio --yes

Found existing installation: torch 2.4.0
Uninstalling torch-2.4.0:
  Successfully uninstalled torch-2.4.0
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128


In [2]:
!pip install ninja packaging setuptools wheel

In [3]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu130

Looking in indexes: https://download.pytorch.org/whl/nightly/cu130


In [4]:
!git clone https://github.com/state-spaces/mamba.git
!pip install ./mamba --no-build-isolation

fatal: destination path 'mamba' already exists and is not an empty directory.
Processing ./mamba
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.0/177.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 MB 14.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 20.8 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 MB 13.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.3/29.3 MB 20.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.8/897.8 kB 36.5 MB/s eta 0:00:00
  Created wheel for mamba_ssm: filename=mamba_ssm-2.3.1-cp312-cp312-linux_x86_64.whl size=533592144 sha256=d66a3c4c94a693e02d341cace7a6af0b72177b6afa655a25e3a6505130a68cbf
  Stored in directory: /tmp/pip-ephem-wheel-cache-li53c0wo/wheels/c5/65/cd/cbb8861ebf805017ad3e607288a78ab08585b0c718394d4fad
Successfully built mamba_ssm


In [ ]:
import torch
from mamba_ssm import Mamba
print(f"Mamba installed successfully. CUDA available: {torch.cuda.is_available()}")


In [ ]:
!pip install causal-conv1d>=1.4.0 --no-build-isolation

: 

In [ ]:
!pip install mamba-ssm --no-build-isolation

In [4]:

!pip install verbose causal-conv1d>=1.4.0 mamba-ssm --no-build-isolation 

ERROR: Operation cancelled by user


In [15]:

print("Loading model...", flush=True)
model = AutoModelForCausalLM.from_pretrained(
	local_model_dir,
	torch_dtype=torch_dtype,
	trust_remote_code=True,
	device_map=DEVICE_MAP,
)


Loading model...


`torch_dtype` is deprecated! Use `dtype` instead!


ImportError: mamba-ssm is required by the Mamba model but cannot be imported

In [1]:
print('test')

test
